# Insturctions to follow before executing this script

## Step 1: Create the Isolated phase1_ltx (Run in Terminal)



### Tunnel for Jupyter
ssh -f -N -L 5402:scn33-mn:8888 rahkus@login.npsf.cdac.in
### Tunnel for Gradio
ssh -f -N -L 7862:scn33-mn:7860 rahkus@login.npsf.cdac.in


In [1]:
# 1. THE FROZEN RULE: Core Config First
import core_config
print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
from core_config import SYSTEM_DIRS

# 2. ISOLATE GPU: Lock this entire notebook to a specific GPU to prevent VRAM collisions
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" # Change to 0, 1, 2, or 3
print(f"[SYSTEM] Notebook strictly locked to GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.jupyter/lab/workspaces
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.


In [2]:
!kill 9 2386856

/bin/bash: line 1: kill: (9) - No such process
/bin/bash: line 1: kill: (2386856) - No such process


In [3]:
!nvidia-smi

Tue Aug 11 16:13:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:07:00.0 Off |                    0 |
| N/A   28C    P0             54W /  400W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
!pip install --upgrade jupyter ipywidgets

In [5]:
import sys
import subprocess

print("Installing FLUX.2 Diffusers Bridge with a compatible Hub pin...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-U",
    "git+https://github.com/huggingface/diffusers.git",
    "huggingface-hub==0.36.0",
    "transformers==4.56.1",
], check=True)

# Fail at setup time if package constraints are unresolved; don't leave this
# until a user clicks Generate and receives a long lazy-import traceback.
subprocess.run([
    sys.executable, "-c",
    (
        "import importlib.metadata as m; "
        "from packaging.version import Version; "
        "hub=Version(m.version('huggingface-hub')); "
        "assert Version('0.34.0') <= hub < Version('1.0.0'), hub; "
        "from diffusers import DiffusionPipeline, Flux2KleinPipeline; "
        "print('FLUX_IMPORT_CHECK=PASS', 'huggingface_hub='+str(hub), "
        "'diffusers='+m.version('diffusers'))"
    ),
], check=True)
print("FLUX.2 dependencies installed and import-checked.")

from packaging.version import Version
_loaded_hub = sys.modules.get("huggingface_hub")
if _loaded_hub is not None:
    _loaded_hub_version = Version(getattr(_loaded_hub, "__version__", "0"))
    if not (Version("0.34.0") <= _loaded_hub_version < Version("1.0.0")):
        raise RuntimeError(
            f"FLUX dependencies were updated on disk, but this Jupyter kernel still "
            f"has huggingface-hub {_loaded_hub_version} loaded in memory. "
            "Restart the kernel, then execute the notebook again from the top. "
            "Do not continue to TRELLIS/FLUX inference in this stale kernel."
        )


1. Installing PyTorch 2.6.0 (CUDA 12.4)...
Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
Looking in indexes: https://download.pytorch.org/whl/cu124
2. Installing ML and 3D dependencies...
Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install imageio imageio-ffmpeg tqdm easydict opencv-python-headless ninja trimesh transformers==4.56.1 gradio tensorboard pandas lpips zstandard kornia timm einops safetensors fire accelerate ipywidgets
Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install -U git+https://github.com/huggingface/diffusers.git
  Cloning https://github.com/huggingface/diffusers.git to ./aireos_cache/pip-req-build-9aozzycf


  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /nlsasfs/home/lmedge/rahkus/aireos_cache/pip-req-build-9aozzycf


  Resolved https://github.com/huggingface/diffusers.git to commit 90c0ffdc045902a3667d473d2fbfc03e8716dba9
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 13.3 MB/s  0:00:00
  Created wheel for diffusers: filename=diffusers-0.40.0.dev0-py3-none-any.whl size=5685833 sha256=24b94601aefa53fee6155f5e0af5a5d8b6d97e430e8ef6d88c89b295fcbac787
  Stored in directory: /nlsasfs/home/lmedge/rahkus/aireos_cache/pip-ephem-wheel-cache-57j9ohui/wheels/4d/b7/a8/6f9549ceec5daad78675b857ac57d697c387062506520a7b50
Successfully built diffusers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.56.1 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.27.0 which is incompatible.


Running: /nlsasfs/home/lmedge/rahkus/Conda/envs/phase2_3d/bin/python -m pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
  Cloning https://github.com/EasternJournalist/utils3d.git (to revision 9a4eb15e4021b67b12c460c7057d642626897ec8) to ./aireos_cache/pip-req-build-ygqfgyyq


  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d.git /nlsasfs/home/lmedge/rahkus/aireos_cache/pip-req-build-ygqfgyyq
  Running command git rev-parse -q --verify 'sha^9a4eb15e4021b67b12c460c7057d642626897ec8'
  Running command git fetch -q https://github.com/EasternJournalist/utils3d.git 9a4eb15e4021b67b12c460c7057d642626897ec8
  Running command git checkout -q 9a4eb15e4021b67b12c460c7057d642626897ec8


... (15 outputs left out)

  Running command git submodule update --init --recursive -q


  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/JeffreyXiang/FlexGEMM.git to ./aireos_cache/pip-install-0fxg4h40/flex-gemm_3b67cee95a41413395243c938cdc4a80


  Running command git clone --filter=blob:none --quiet https://github.com/JeffreyXiang/FlexGEMM.git /nlsasfs/home/lmedge/rahkus/aireos_cache/pip-install-0fxg4h40/flex-gemm_3b67cee95a41413395243c938cdc4a80


  Resolved https://github.com/JeffreyXiang/FlexGEMM.git to commit 6dd94a859c26ee8246888502eada3dd8ad85532e
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for o_voxel: filename=o_voxel-0.0.1-cp310-cp310-linux_x86_64.whl size=253124 sha256=a0985e6bd75899d7a7bf4d3ebce7c71bceab69217b1ba6e2eaa4de0449f651d5
  Stored in directory: /nlsasfs/home/lmedge/rahkus/.cache/pip/wheels/8b/5d/ac/0e40d8d80233efe5b781a1cb449907578af08076cba54a92bc
Successfully built o_voxel
  Attempting uninstall: o_voxel
    Found existing installation: o_voxel 0.0.1
    Uninstalling o_voxel-0.0.1:
      Successfully uninstalled o_voxel-0.0.1
Environment Ready! TRELLIS.2 & FLUX.2 successfully installed and mapped.


In [6]:
import core_config
from core_config import SYSTEM_DIRS
from huggingface_hub import snapshot_download
try:
    from huggingface_hub.errors import LocalEntryNotFoundError
except ImportError:
    from huggingface_hub.utils import LocalEntryNotFoundError

import os
import torch
import random
import gc
import gradio as gr
from diffusers import DiffusionPipeline

from purge_manager import DiskPurgeManager
from storage_manager import managed_storage_operation
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "images")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def load_flux_pipeline():
    model_id = "black-forest-labs/FLUX.2-klein-4B"
    hub_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")
    kwargs = dict(
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
        text_encoder_2=None,
        tokenizer_2=None,
        feature_extractor=None,
        image_encoder=None,
    )

    try:
        local_model_dir = snapshot_download(
            repo_id=model_id,
            cache_dir=hub_cache_dir,
            local_files_only=True,
        )
        print(f"FLUX cache hit; resolved local snapshot: {local_model_dir}")
    except LocalEntryNotFoundError as cache_error:
        # Only a true local cache miss enters storage admission. Don't mask import,
        # compatibility, model-config, or CUDA errors as missing model files.
        required_gb = (
            estimate_hf_repo_bytes(model_id) / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        print(
            f"FLUX cache miss ({type(cache_error).__name__}). "
            f"Storage admission budget: {required_gb:.2f} GB"
        )
        with managed_storage_operation(
            "models",
            required_gb=required_gb,
            require_fresh_quota=True,
            minimum_free_gb=float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0")),
        ):
            local_model_dir = snapshot_download(
                repo_id=model_id,
                cache_dir=hub_cache_dir,
            )

    pipe = DiffusionPipeline.from_pretrained(local_model_dir, **kwargs)
    pipe.enable_model_cpu_offload()
    return pipe
flux_pipe = load_flux_pipeline()

print("\nExecuting VRAM Load & Purge...")
DiskPurgeManager.purge_hf_model("black-forest-labs/FLUX.2-klein-4B")

@torch.inference_mode()
def generate_concept(prompt: str, seed: int, randomize: bool):
    if randomize: seed = random.randint(0, 999999)
    print(f"\n[GENERATING] Prompt: '{prompt}' | Seed: {seed}")
    
    generator = torch.Generator(device="cpu").manual_seed(seed)
    image = flux_pipe(
        prompt,
        guidance_scale=3.5, 
        num_inference_steps=28, 
        max_sequence_length=512,
        generator=generator
    ).images[0].convert("RGB")
    
    out_path = os.path.join(OUTPUT_DIR, f"flux2_concept_{seed}.png")
    with managed_storage_operation(
        "outputs",
        required_gb=0.05,
        require_fresh_quota=False,
        minimum_free_gb=0.02,
    ):
        image.save(out_path)
    
    gc.collect()
    torch.cuda.empty_cache()
    return image, seed

with gr.Blocks(title="FLUX.2 Standalone", theme=gr.themes.Monochrome()) as demo_flux:
    gr.Markdown("# 🎨 FLUX.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", value="A sci-fi crate", lines=4)
            seed = gr.Number(label="Seed", value=42, precision=0)
            randomize = gr.Checkbox(label="Randomize", value=True)
            generate_btn = gr.Button("Generate", variant="primary")
        with gr.Column(scale=2):
            output_image = gr.Image(label="FLUX.2 Output")

    generate_btn.click(fn=generate_concept, inputs=[prompt, seed, randomize], outputs=[output_image, seed])

print("Starting FLUX Server on port 7863...")
demo_flux.launch(server_name="0.0.0.0", server_port=7863, inline=True, share=False)

RuntimeError: Failed to import diffusers.pipelines.pipeline_utils because of the following error (look up to see its traceback):
Failed to import diffusers.models.autoencoders.autoencoder_kl because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.peft because of the following error (look up to see its traceback):
huggingface-hub>=0.34.0,<1.0 is required for a normal functioning of this module, but found huggingface-hub==1.27.0.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [ ]:
# demo.close()

In [ ]:
!nvidia-smi